In [1]:
import pandas as pd
import numpy as np
from statsmodels.stats.proportion import proportions_ztest

df = pd.read_csv("../data/cookie_cats.csv")
rng = np.random.default_rng(1)
y = df["retention_7"].to_numpy().astype(int)
n40 = (df.version == "gate_40").sum()
n = len(df)

pvals = []
for _ in range(2000):
    idx = rng.permutation(n)
    a, b = y[idx[:n40]], y[idx[n40:]]
    pvals.append(proportions_ztest([a.sum(), b.sum()], [len(a), len(b)])[1])

pvals = np.array(pvals)
print("доля p < 0,05:", (pvals < 0.05).mean())

доля p < 0,05: 0.0525


In [2]:
rng = np.random.default_rng(2)
checkpoints = [int(n * k / 14) for k in range(1, 15)]
stopped = 0

for _ in range(1000):
    yy = rng.permutation(y)
    g = rng.random(n) < 0.5
    for m in checkpoints:
        a, b = yy[:m][g[:m]], yy[:m][~g[:m]]
        if proportions_ztest([a.sum(), b.sum()], [len(a), len(b)])[1] < 0.05:
            stopped += 1
            break

print("доля остановок с ложной значимостью:", stopped / 1000)

доля остановок с ложной значимостью: 0.221


In [3]:
rng = np.random.default_rng(3)
p0 = df[df.version == "gate_30"].retention_7.mean()
n1, n2 = 44700, 45489

for delta in [0.003, 0.005, 0.0074, 0.01]:
    hits = 0
    for _ in range(1000):
        a = rng.random(n1) < p0
        b = rng.random(n2) < p0 + delta
        if proportions_ztest([a.sum(), b.sum()], [n1, n2])[1] < 0.05:
            hits += 1
    print("эффект", delta * 100, "п.п., мощность:", hits / 1000)

эффект 0.3 п.п., мощность: 0.211
эффект 0.5 п.п., мощность: 0.457
эффект 0.74 п.п., мощность: 0.82
эффект 1.0 п.п., мощность: 0.968
